In [1]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error
import pickle

import xgboost as xgb

In [2]:
import mlflow
import os


db_path = os.path.abspath("/workspaces/MLOPS/00-mlflow/mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("nyc-taxi-exp")

#mlflow ui --backend-store-uri sqlite:///mlflow.db

<Experiment: artifact_location='/workspaces/MLOPS/03-Experiment_tracking/mlruns/2', creation_time=1788872133033, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1788872133033, lifecycle_stage='active', name='nyc-taxi-exp', tags={}, trace_location=None, workspace='default'>

In [3]:
def read_dataframe(path):
    df=pd.read_parquet(path)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    
    df=df[((df.duration>=1) & (df.duration<=60))]
    
    categorical = ['PULocationID', 'DOLocationID']
    
    df[categorical]=df[categorical].astype(str)
    
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    
    return df

In [4]:
df_train=read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-01.parquet')
df_val=read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-02.parquet')

In [5]:
len(df_train), len(df_val)

(38088, 35319)

In [6]:
categorical = ['PU_DO']
numerical = ['trip_distance']

dv=DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [7]:
target='duration'
y_train= df_train[target].values
y_val= df_val[target].values

In [8]:
mlflow.xgboost.autolog(disable=False)

In [9]:
with mlflow.start_run():
    
    train=xgb.DMatrix(X_train, label=y_train)
    valid=xgb.DMatrix(X_val, label=y_val)
    
    best_params={
    "learning_rate": 0.07514278112841959,
    "max_depth": 6,
    "min_child_weight": 9.028786614533509,
    "objective": "reg:squarederror",
    "reg_alpha": 0.04080737438776132,
    "reg_lambda": 0.04628904701264794,
    "seed":42,
    }
    
    mlflow.log_params(best_params)
    
    booster = xgb.train(
                params=best_params,
                dtrain=train,
                num_boost_round=30,
                evals=[(valid, "validation")],
                early_stopping_rounds=50,
                verbose_eval=100
            )

    y_pred = booster.predict(valid)
    rmse=root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse",rmse)
    
    
    with open("models/preprocessor.b","wb") as f_out:
        pickle.dump(dv, f_out)
    
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")
    
    mlflow.xgboost.log_model(booster, name="models_mlflow")
    

[0]	validation-rmse:9.89404
[29]	validation-rmse:6.26567


2026/09/18 14:27:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [15]:
# Replace with your actual model URI (e.g., "runs:/<run_id>/my_model" or "models:/<model_name>/1")
model_uri = "runs:/cd2e03335b6f4450b98eafc7fa89a117/models_mlflow"
loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(valid)
rmse=root_mean_squared_error(y_val, y_pred)
print(rmse)

5.765722147877208
